# 05️⃣ Risk Assessment

Aggregate daily returns correctly and compound portfolios. Compute annualized return, volatility, Sharpe ratio, Sortino ratio, and Maximum Drawdown.

In [1]:
import os, pathlib, pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')


In [2]:
NOTEBOOK_DIR = pathlib.Path(os.path.abspath('') if '__file__' not in locals() else os.path.dirname(__file__))
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PARQUET = PROJECT_ROOT / 'data' / 'parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PARQUET / 'portfolio_construction.parquet')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# Debt rate: 5% annualized
r_debt = 0.05 / 252

# Daily returns
df['Return_Conservative'] = df['Conservative_Equity'] * df['Passive_Equity_Return'] + df['Conservative_Debt'] * r_debt - df['Conservative_Transaction_Cost']
df['Return_Balanced'] = df['Balanced_Equity'] * df['Active_Equity_Return'] + df['Balanced_Debt'] * r_debt - df['Balanced_Transaction_Cost']
df['Return_Aggressive'] = df['Aggressive_Equity'] * df['Active_Equity_Return'] + df['Aggressive_Debt'] * r_debt - df['Aggressive_Transaction_Cost']

profiles = ['Conservative', 'Balanced', 'Aggressive']
metrics = []

for p in profiles:
    ret_col = f'Return_{p}'
    
    # Compounding step-by-step with capital preservation check
    daily_rets = df[ret_col].values.copy()
    cum_vals = np.zeros(len(daily_rets))
    curr_val = 1.0
    for idx, r in enumerate(daily_rets):
        curr_val *= (1.0 + r)
        if curr_val <= 0.05: # Bankruptcy floor at 95% drawdown
            curr_val = max(curr_val, 0.0)
            daily_rets[idx:] = 0.0
            cum_vals[idx:] = curr_val
            break
        cum_vals[idx] = curr_val
        
    df[f'CumValue_{p}'] = cum_vals
    
    # Annualized volatility
    vol = daily_rets.std() * np.sqrt(252)
    
    # Annualized return (CAGR)
    n_days = len(df)
    end_val = cum_vals[-1]
    cagr = (end_val / 1.0) ** (252 / n_days) - 1.0
    
    # Sharpe Ratio
    rf_annual = 0.05
    sharpe = (cagr - rf_annual) / (vol + 1e-9)
    
    # Sortino Ratio
    downside_returns = daily_rets[daily_rets < 0]
    downside_std = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 1e-9
    sortino = (cagr - rf_annual) / (downside_std + 1e-9)
    
    # Max Drawdown
    running_max = np.maximum.accumulate(cum_vals)
    drawdowns = (cum_vals - running_max) / (running_max + 1e-9)
    max_dd = drawdowns.min()
    
    metrics.append({
        'Profile': p,
        'Annualized_Return': cagr,
        'Volatility': vol,
        'Sharpe_Ratio': sharpe,
        'Sortino_Ratio': sortino,
        'Max_Drawdown': max_dd
    })

risk_df = pd.DataFrame(metrics)
print(risk_df)
risk_df.to_csv(OUTPUT_DIR / 'portfolio_risk_metrics.csv', index=False)
df.to_parquet(DATA_PARQUET / 'portfolio_risk_results.parquet', index=False)

# Plot cumulative returns comparison
plt.figure(figsize=(12, 6))
for p in profiles:
    cagr_val = risk_df[risk_df['Profile'] == p]['Annualized_Return'].iloc[0]
    mdd_val = risk_df[risk_df['Profile'] == p]['Max_Drawdown'].iloc[0]
    plt.plot(df['Date'], df[f'CumValue_{p}'], label=f"{p} (CAGR: {cagr_val:.2%}, MDD: {mdd_val:.2%})")

# Plot NIFTY50 passive benchmark (100% passive equity)
nifty_cum = (1 + df['Passive_Equity_Return']).cumprod()
nifty_cagr = (nifty_cum.iloc[-1]) ** (252 / len(df)) - 1.0
plt.plot(df['Date'], nifty_cum, 'k--', label=f"NIFTY50 Index (CAGR: {nifty_cagr:.2%})")

plt.title('Out-of-Sample Portfolio Growth (2016-2021) - Net of Fees & Slippage')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (Base 1.0)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'portfolio_cumulative_returns.png', dpi=300)
plt.close()
print('Risk assessment complete. Cumulative returns chart generated.')


        Profile  Annualized_Return  Volatility  Sharpe_Ratio  Sortino_Ratio  \
0  Conservative           0.066461    0.037928      0.433995       0.501133   
1      Balanced           0.051826    0.114129      0.015999       0.020445   
2    Aggressive           0.041410    0.195229     -0.044000      -0.054881   

   Max_Drawdown  
0     -0.085996  
1     -0.259739  
2     -0.424238  


Risk assessment complete. Cumulative returns chart generated.
